# ReadWise AI — ETDD70 Dyslexia Risk Screening Training

This notebook trains and benchmarks dyslexia-risk screening models on the **ETDD70 dataset** ([Zenodo Record 13332134](https://zenodo.org/records/13332134)).

### Methodological Guarantees
1. **Screening, Not Diagnosis**: Outputs are screening indicators for educational intervention, not diagnostic judgments.
2. **Subject-Level Leakage Prevention**: We enforce `GroupKFold` on `subject_id` so that multiple reading trials of the same child never cross train and validation folds.
3. **Model Benchmarking**: Compares Logistic Regression, Random Forest, and XGBoost.
4. **Transparent Explainability**: Incorporates SHAP values to explain feature contribution for each assessment.

In [ ]:
# 1. Install & Import Libraries
!pip install -q xgboost shap scikit-learn pandas numpy matplotlib seaborn

import os, glob, zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
import shap

In [ ]:
# 2. Download ETDD70 from Zenodo
DATA_DIR = './etdd70'
os.makedirs(DATA_DIR, exist_ok=True)

LABEL_URL = 'https://zenodo.org/records/13332134/files/dyslexia_class_label.csv?download=1'
DATA_URL = 'https://zenodo.org/records/13332134/files/data.zip?download=1'

if not os.path.exists(f'{DATA_DIR}/dyslexia_class_label.csv'):
    print('Downloading dyslexia_class_label.csv...')
    !curl -L -o {DATA_DIR}/dyslexia_class_label.csv '{LABEL_URL}'

if not os.path.exists(f'{DATA_DIR}/extracted'):
    print('Downloading data.zip (~127MB)...')
    !curl -L -o {DATA_DIR}/data.zip '{DATA_URL}'
    print('Extracting data.zip...')
    with zipfile.ZipFile(f'{DATA_DIR}/data.zip', 'r') as zf:
        zf.extractall(f'{DATA_DIR}/extracted')
    print('Extracted successfully!')

In [ ]:
# 3. Aggregate Trial Metrics and Join Labels
labels_df = pd.read_csv(f'{DATA_DIR}/dyslexia_class_label.csv')
metrics_files = glob.glob(f'{DATA_DIR}/extracted/data/*_metrics.csv')
print(f'Found {len(metrics_files)} trial metrics CSVs for {len(labels_df)} subjects.')

trial_rows = []
for f in metrics_files:
    df = pd.read_csv(f)
    trial_info = df.iloc[0][[
        'sid', 'task', 'n_fix_trial', 'sum_fix_dur_trial', 'dwell_time_trial',
        'mean_fix_dur_trial', 'n_sacc_trial', 'sum_sacc_dur_trial',
        'mean_sacc_dur_trial', 'mean_sacc_ampl_trial', 'ratio_progress_regress_trial',
        'n_regress_trial', 'n_progress_trial'
    ]].to_dict()
    trial_rows.append(trial_info)

trials_df = pd.DataFrame(trial_rows)
dataset = pd.merge(trials_df, labels_df, left_on='sid', right_on='subject_id')
dataset['risk_label'] = dataset['class_id']
print(f'Processed dataset shape: {dataset.shape}')
dataset.head()

In [ ]:
# 4. GroupKFold Cross-Validation (Zero Subject-Level Leakage)
feature_cols = [
    'n_fix_trial', 'sum_fix_dur_trial', 'dwell_time_trial',
    'mean_fix_dur_trial', 'n_sacc_trial', 'sum_sacc_dur_trial',
    'mean_sacc_dur_trial', 'mean_sacc_ampl_trial', 'ratio_progress_regress_trial',
    'n_regress_trial', 'n_progress_trial'
]

X = dataset[feature_cols]
y = dataset['risk_label']
groups = dataset['subject_id']

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, random_state=42))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=6, random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100, max_depth=4, eval_metric='logloss', random_state=42
    )
}

results = {name: {'auc': [], 'acc': []} for name in models}
gkf = GroupKFold(n_splits=5)

print('Evaluating models with 5-Fold GroupKFold...')
for name, model in models.items():
    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_te)[:, 1]
        preds = model.predict(X_te)
        results[name]['auc'].append(roc_auc_score(y_te, probs))
        results[name]['acc'].append(accuracy_score(y_te, preds))

print('='*70)
for name, res in results.items():
    print(f'{name:22s} | ROC-AUC: {np.mean(res["auc"]):.3f} +/- {np.std(res["auc"]):.3f} | Accuracy: {np.mean(res["acc"]):.3f} +/- {np.std(res["acc"]):.3f}')
print('='*70)

In [ ]:
# 5. Fit Final Random Forest Model & SHAP Summary Plot
rf_final = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_final.fit(X, y)

explainer = shap.TreeExplainer(rf_final)
shap_values = explainer.shap_values(X)

if hasattr(shap_values, 'ndim') and shap_values.ndim == 3:
    class1_shap = shap_values[:, :, 1]
elif isinstance(shap_values, list):
    class1_shap = shap_values[1]
else:
    class1_shap = shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(class1_shap, X, show=True)